In [0]:
from pyspark.sql.functions import current_timestamp
import xml.etree.ElementTree as ET

In [0]:

#parse xml file 
tree = ET.parse('/Volumes/skztour/00_landing/data_sources/response.xml')
root = tree.getroot()

In [0]:
# extract data into a list of dictionaries with proper XML parsing
import json

data = []
for setlist in root:
    record = {
        'setlist_id': setlist.attrib.get('id'),
        'event_date': setlist.attrib.get('eventDate'),
        'last_updated': setlist.attrib.get('lastUpdated'),
        'artist_name': None,
        'artist_mbid': None,
        'venue_name': None,
        'venue_id': None,
        'venue_city': None,
        'sets': None,
        'info': None,
        'url': None
    }
    
    for elem in setlist:
        if elem.tag == 'artist':
            record['artist_name'] = elem.attrib.get('name')
            record['artist_mbid'] = elem.attrib.get('mbid')
        elif elem.tag == 'venue':
            record['venue_name'] = elem.attrib.get('name')
            record['venue_id'] = elem.attrib.get('id')
            # Get city from nested element
            city_elem = elem.find('city')
            if city_elem is not None:
                record['venue_city'] = city_elem.attrib.get('name')
        elif elem.tag == 'sets':
            # Store sets as JSON string for now (can parse further in silver)
            sets_data = []
            for set_elem in elem.findall('set'):
                songs = []
                for song in set_elem.findall('song'):
                    songs.append(song.attrib.get('name'))
                sets_data.append({'songs': songs})
            record['sets'] = json.dumps(sets_data) if sets_data else None
        elif elem.tag == 'info':
            record['info'] = elem.text
        elif elem.tag == 'url':
            record['url'] = elem.text
    
    data.append(record)


In [0]:
# convert into a dataframe and save to bronze table
from pyspark.sql.types import StructType, StructField, StringType

# Define schema with properly extracted fields
schema = StructType([
    StructField("setlist_id", StringType(), True),
    StructField("event_date", StringType(), True),
    StructField("last_updated", StringType(), True),
    StructField("artist_name", StringType(), True),
    StructField("artist_mbid", StringType(), True),
    StructField("venue_name", StringType(), True),
    StructField("venue_id", StringType(), True),
    StructField("venue_city", StringType(), True),
    StructField("sets", StringType(), True),
    StructField("info", StringType(), True),
    StructField("url", StringType(), True)
])

df = spark.createDataFrame(data, schema=schema)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("skztour.01_bronze.raw_setlist_fm")

In [0]:
df.show()

+--------------+--------------+----+--------------+--------------------+--------------------+
|        artist|         venue|tour|          sets|                info|                 url|
+--------------+--------------+----+--------------+--------------------+--------------------+
|\n            |\n            |NULL|\n            |Concert entirely ...|https://www.setli...|
|\n            |\n            |NULL|\n            |                NULL|https://www.setli...|
|\n            |\n            |NULL|\n            |"STAY in Our Litt...|https://www.setli...|
|\n            |\n            |NULL|\n            |"STAY in Our Litt...|https://www.setli...|
|\n            |\n            |NULL|\n            |"STAY in Our Litt...|https://www.setli...|
|\n            |\n            |NULL|\n            |                NULL|https://www.setli...|
|\n            |\n            |NULL|\n            |                NULL|https://www.setli...|
|\n            |\n            |NULL|\n            |         

In [0]:
display(df)

setlist_id,event_date,last_updated,artist_name,artist_mbid,venue_name,venue_id,venue_city,sets,info,url
2b4d4ce6,06-06-2026,2026-06-08T11:10:41.944+0000,Stray Kids,142b343d-bf5a-428c-a64f-6d1a7566bbe9,Flushing Meadows Corona Park,43d2830b,Queens,"[{""songs"": [""SKZ Anthem"", ""TOPLINE"", ""S-Class"", ""Bounce Back"", ""Ment 1"", ""MANIAC"", ""DOMINO"", ""Thunderous"", ""Band performance"", ""DIVINE"", ""Walkin on Water"", ""God's Menu"", ""Ment 2"", ""Chk Chk Boom"", ""ITEM"", ""LALALALA"", ""Social Path"", ""Ment 3"", ""Side Effects"", ""Do It"", ""CEREMONY"", ""Blind Spot"", ""MIROH""]}]","Concert entirely played with live band except from ""Side Effects"" to ""CEREMONY"" and ""MIROH"". ""Stray Kids"" was on printed setlist but wasn't played. Seungmin absent due to injury.",https://www.setlist.fm/setlist/stray-kids/2026/flushing-meadows-corona-park-queens-ny-2b4d4ce6.html
234bc47b,05-04-2026,2026-04-05T14:09:30.475+0000,Stray Kids,142b343d-bf5a-428c-a64f-6d1a7566bbe9,Inspire Arena,3d12183,Incheon,"[{""songs"": [""DIVINE"", ""Half Time"", ""Bounce Back"", ""Collision"", ""STAY"", ""Burnin' Tires"", ""Truman"", ""ESCAPE"", ""CINEMA"", ""Give Me Your TMI"", ""CASE 143"", ""Do It"", ""CEREMONY"", ""0801"", ""In My Head"", ""Haven"", ""Social Path"", ""MIROH"", ""TA"", ""Star Lost"", ""Chk Chk Boom""]}]",null,https://www.setlist.fm/setlist/stray-kids/2026/inspire-arena-incheon-south-korea-234bc47b.html
1b4bc584,04-04-2026,2026-04-04T13:13:17.175+0000,Stray Kids,142b343d-bf5a-428c-a64f-6d1a7566bbe9,Inspire Arena,3d12183,Incheon,"[{""songs"": [""DIVINE"", ""Half Time"", ""Bounce Back"", ""Collision"", ""STAY"", ""Burnin' Tires"", ""CINEMA"", ""ESCAPE"", ""Truman"", ""Give Me Your TMI"", ""CASE 143"", ""Do It"", ""CEREMONY"", ""0801"", ""In My Head"", ""Haven"", ""Social Path"", ""MIROH""]}]","""STAY in Our Little House"" Fanmeeting, Day 3",https://www.setlist.fm/setlist/stray-kids/2026/inspire-arena-incheon-south-korea-1b4bc584.html
134bc585,29-03-2026,2026-04-04T13:13:02.975+0000,Stray Kids,142b343d-bf5a-428c-a64f-6d1a7566bbe9,Inspire Arena,3d12183,Incheon,"[{""songs"": [""DIVINE"", ""Half Time"", ""Bounce Back"", ""Collision"", ""STAY"", ""Burnin' Tires"", ""Truman"", ""CINEMA"", ""ESCAPE"", ""Give Me Your TMI"", ""CASE 143"", ""Do It"", ""CEREMONY"", ""0801"", ""I Like It"", ""Phoenix"", ""Social Path"", ""MIROH""]}]","""STAY in Our Little House"" Fanmeeting, Day 2",https://www.setlist.fm/setlist/stray-kids/2026/inspire-arena-incheon-south-korea-134bc585.html
b4bc586,28-03-2026,2026-04-04T13:12:43.364+0000,Stray Kids,142b343d-bf5a-428c-a64f-6d1a7566bbe9,Inspire Arena,3d12183,Incheon,"[{""songs"": [""DIVINE"", ""Half Time"", ""Bounce Back"", ""Collision"", ""STAY"", ""Truman"", ""Burnin' Tires"", ""ESCAPE"", ""CINEMA"", ""Give Me Your TMI"", ""CASE 143"", ""Do It"", ""CEREMONY"", ""0801"", ""I Like It"", ""Phoenix"", ""Social Path"", ""MIROH""]}]","""STAY in Our Little House"" Fanmeeting, Day 1",https://www.setlist.fm/setlist/stray-kids/2026/inspire-arena-incheon-south-korea-b4bc586.html
3b4d481c,22-01-2026,2026-02-02T02:21:20.964+0000,Stray Kids,142b343d-bf5a-428c-a64f-6d1a7566bbe9,Paris La Défense Arena,1bd265e0,Nanterre,"[{""songs"": [""Intro"", ""Chk Chk Boom"", ""Do It"", ""CEREMONY"", ""LALALALA""]}]",null,https://www.setlist.fm/setlist/stray-kids/2026/paris-la-defense-arena-nanterre-france-3b4d481c.html
4b4cf70e,10-01-2026,2026-01-24T19:54:01.592+0000,Stray Kids,142b343d-bf5a-428c-a64f-6d1a7566bbe9,Taipei Dome,3de91c3,Taipei,"[{""songs"": [""DIVINE"", ""Do It"", ""CEREMONY""]}]",null,https://www.setlist.fm/setlist/stray-kids/2026/taipei-dome-taipei-taiwan-4b4cf70e.html
b4da5ee,31-12-2025,2025-12-31T15:58:53.173+0000,Stray Kids,142b343d-bf5a-428c-a64f-6d1a7566bbe9,MBC Dream Center,5bd11b38,Goyang-si,"[{""songs"": [""Thunderous"", ""DIVINE""]}]",null,https://www.setlist.fm/setlist/stray-kids/2025/mbc-dream-center-goyang-si-south-korea-b4da5ee.html
2b4d24a2,29-12-2025,2025-12-30T04:16:17.731+0000,Stray Kids,142b343d-bf5a-428

Databricks data profile. Run in Databricks to view.

In [0]:
# Inspect the actual XML structure to see what we might be missing
print("Root tag:", root.tag)
print("Root attributes:", root.attrib)
print("Number of children:", len(root))

print("\n=== First child element structure ===")
if len(root) > 0:
    first_child = root[0]
    print(f"Child tag: {first_child.tag}")
    print(f"Child attributes: {first_child.attrib}")
    print(f"Number of sub-elements: {len(first_child)}")
    
    print("\n--- Sub-elements ---")
    for elem in first_child:
        print(f"  Tag: {elem.tag}")
        print(f"  Attributes: {elem.attrib}")
        print(f"  Text: {repr(elem.text)}")
        print(f"  Has children: {len(elem) > 0}")
        if len(elem) > 0:
            print(f"  Children tags: {[child.tag for child in elem]}")
        print()

Root tag: setlists
Root attributes: {'itemsPerPage': '20', 'page': '1', 'total': '678'}
Number of children: 20

=== First child element structure ===
Child tag: setlist
Child attributes: {'id': '2b4d4ce6', 'versionId': 'g6b1b024a', 'eventDate': '06-06-2026', 'lastUpdated': '2026-06-08T11:10:41.944+0000'}
Number of sub-elements: 5

--- Sub-elements ---
  Tag: artist
  Attributes: {'mbid': '142b343d-bf5a-428c-a64f-6d1a7566bbe9', 'name': 'Stray Kids', 'sortName': 'Stray Kids', 'disambiguation': 'South Korean boy group'}
  Text: '\n            '
  Has children: True
  Children tags: ['url']

  Tag: venue
  Attributes: {'id': '43d2830b', 'name': 'Flushing Meadows Corona Park'}
  Text: '\n            '
  Has children: True
  Children tags: ['city', 'url']

  Tag: sets
  Attributes: {}
  Text: '\n            '
  Has children: True
  Children tags: ['set']

  Tag: info
  Attributes: {}
  Text: 'Concert entirely played with live band except from "Side Effects" to "CEREMONY" and "MIROH". "Stray 

In [0]:
# Inspect the actual XML structure to see what we might be missing
print("Root tag:", root.tag)
print("Root attributes:", root.attrib)
print("Number of children:", len(root))

print("\n=== First child element structure ===")
if len(root) > 0:
    first_child = root[0]
    print(f"Child tag: {first_child.tag}")
    print(f"Child attributes: {first_child.attrib}")
    print(f"Number of sub-elements: {len(first_child)}")
    
    print("\n--- Sub-elements ---")
    for elem in first_child:
        print(f"  Tag: {elem.tag}")
        print(f"  Attributes: {elem.attrib}")
        print(f"  Text: {repr(elem.text)}")
        print(f"  Has children: {len(elem) > 0}")
        if len(elem) > 0:
            print(f"  Children tags: {[child.tag for child in elem]}")
        print()